In [ ]:
import numpy as np
from pdbfixer import PDBFixer
try:
    import openmm as mm
    import openmm.app as app
    import openmm.unit as unit
except ImportError:
    import simtk.openmm as mm
    import simtk.openmm.app as app
    import simtk.unit as unit
from tqdm import tqdm
import glob
import sys
import os
sys.path.insert(0, '../../')
from openabc.utils.CA2AA import split_protein, combine_proteins

os.makedirs('ca-tmp', exist_ok=True)
os.makedirs('aa-tmp', exist_ok=True)

This example demonstrates converting CA model to all-atom model with `pdbfixer`. `pdbfixer` can be installed via `conda` and provides powerful, Python-native APIs for PDB structure completion.

**Important**: These converted structures have not been validated for stability; users should perform thorough structural relaxation (e.g., energy minimization) before running all-atom simulations."

In [ ]:
# convert the system composed of HP1alpha dimers from CA to all-atom representation
# separate each dimer into individual pdb files, fix them, then merge
# here for speed, we only convert the first 5 dimers
ca_pdb = 'hp1alpha_100_dimers_CA.pdb'
split_protein(ca_pdb, num_chains=[2 * 100], num_residues=[191], output_path='ca-tmp')

ca_pdb_paths = sorted(glob.glob('ca-tmp/protein*chain*.pdb'))
aa_pdb_paths = []

# convert first 5 dimers, which correspond to first 10 chains
for each_ca_pdb in tqdm(ca_pdb_paths[:10]):
    pdb_basename = os.path.basename(each_ca_pdb)
    fixer = PDBFixer(each_ca_pdb)
    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.removeHeterogens(keepWater=False)
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(pH=7.0)
    each_aa_pdb = f'aa-tmp/{pdb_basename}'
    aa_pdb_paths.append(each_aa_pdb)
    with open(each_aa_pdb, 'w') as f:
        app.PDBFile.writeFile(fixer.topology, fixer.positions, f)

# merge all fixed pdb files into one
combine_proteins(aa_pdb_paths, 'hp1alpha_100_dimers_AA.pdb')